In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_json('data/OTB.jsonl', lines=True)

In [3]:
df = df_og.copy()

NameError: name 'df_og' is not defined

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

# Data Cleaning

In [ ]:
df = df.replace('?',np.nan)

In [ ]:
missing = df.isnull().sum().reset_index(name='Count')
missing['percentage'] = ((df.isnull().sum() / len(df)) * 100).reset_index(name='Percentage')['Percentage']
missing

In [ ]:
pd.Series((pd.Series(df.isnull().sum() / df.shape[0])*100)).plot.bar()

## Drop columns
- TimeControl, Source, Merged due to insufficient data points.
- BlackFideId, WhiteFideId, SourceQuality,  due to no analytical value.


In [ ]:
df[['TimeControl', 'BlackFideId', 'WhiteFideId', 'SourceQuality', 'Source', 'Merged']]

In [ ]:
null_by_sq = df.isnull().assign(SourceQuality=df['SourceQuality']).groupby('SourceQuality').mean() * 100

sns.heatmap(null_by_sq.T, annot=True, fmt='.01f', cmap='YlOrRd')
plt.title('Null Count per Column by SourceQuality')
plt.xlabel('SourceQuality')
plt.show()

In [ ]:
df = df.drop(columns=['WhiteFideId','BlackFideId','TimeControl','Merged','SourceQuality','Source','ImportDate'])

df.columns

## Null Values

In [ ]:
null_count = df.isnull().sum()

null_count

### WhiteTitle and BlackTitle
- the null values are just untitled players and can be replaced with NonTitled

In [ ]:
df['WhiteTitle'] = df['WhiteTitle'].fillna('ungraded')
df['BlackTitle'] = df['BlackTitle'].fillna('ungraded')

In [ ]:
df[['WhiteTitle','BlackTitle']].isnull().sum()

### White and Black
- null values are about 0.1% of whole dataset hence we can drop them

In [ ]:
df[df.White.isnull() | df.Black.isnull()]

In [ ]:
(len(df[df.White.isnull() | df.Black.isnull()]) / len(df)) * 100

In [ ]:
# Drops rows where White or Black is NaN
df.dropna(subset=['White', 'Black'], inplace=True)

### Event and Site
- Most Event happen at same places a lot of times so we sue the most common place to fill the site
- Dropping the 

In [ ]:
df[df['Event'].isnull() | df['Site'].isnull()].shape[0]

In [ ]:
df['Event'] = df['Event'].fillna('Unknown')
df['Site'] = df['Site'].fillna('Unknown')

In [ ]:
df[df['Event'].isnull() | df['Site'].isnull()].shape[0]

### ECO

In [ ]:
eco_data = """{"Moves": "1. b4", "ECO": ["A00"], "Opening": "Polish (Sokolsky) Opening"}
{"Moves": "1. b3", "ECO": ["A01"], "Opening": "Nimzovich-Larsen attack"}
{"Moves": "1. f4", "ECO": ["A02", "A03"], "Opening": "Bird's Opening"}
{"Moves": "1. Nf3", "ECO": ["A04", "A05", "A06", "A07", "A08", "A09"], "Opening": "Reti Opening"}
{"Moves": "1. c4", "ECO": ["A10", "A11", "A12", "A13", "A14", "A15", "A16", "A17", "A18", "A19", "A20", "A21", "A22", "A23", "A24", "A25", "A26", "A27", "A28", "A29", "A30", "A31", "A32", "A33", "A34", "A35", "A36", "A37", "A38", "A39"], "Opening": "English Opening"}
{"Moves": "1. d4", "ECO": ["A40", "A41"], "Opening": "Queen's pawn"}
{"Moves": "1. d4 d6 2. c4 g6 3. Nc3 Bg7 4. e4", "ECO": ["A42"], "Opening": "Modern defence, Averbakh system"}
{"Moves": "1. d4 c5", "ECO": ["A43", "A44"], "Opening": "Old Benoni defence"}
{"Moves": "1. d4 Nf6", "ECO": ["A45", "A46"], "Opening": "Queen's pawn game"}
{"Moves": "1. d4 Nf6 2. Nf3 b6", "ECO": ["A47"], "Opening": "Queen's Indian defence"}
{"Moves": "1. d4 Nf6 2. Nf3 g6", "ECO": ["A48", "A49"], "Opening": "King's Indian, East Indian defence"}
{"Moves": "1. d4 Nf6 2. c4", "ECO": ["A50"], "Opening": "Queen's pawn game"}
{"Moves": "1. d4 Nf6 2. c4 e5", "ECO": ["A51", "A52"], "Opening": "Budapest defence"}
{"Moves": "1. d4 Nf6 2. c4 d6", "ECO": ["A53", "A54", "A55"], "Opening": "Old Indian defence"}
{"Moves": "1. d4 Nf6 2. c4 c5", "ECO": ["A56"], "Opening": "Benoni defence"}
{"Moves": "1. d4 Nf6 2. c4 c5 3. d5 b5", "ECO": ["A57", "A58", "A59"], "Opening": "Benko gambit"}
{"Moves": "1. d4 Nf6 2. c4 c5 3. d5 e6", "ECO": ["A60", "A61", "A62", "A63", "A64", "A65", "A66", "A67", "A68", "A69", "A70", "A71", "A72", "A73", "A74", "A75", "A76", "A77", "A78", "A79"], "Opening": "Benoni defence"}
{"Moves": "1. d4 f5", "ECO": ["A80", "A81", "A82", "A83", "A84", "A85", "A86", "A87", "A88", "A89", "A90", "A91", "A92", "A93", "A94", "A95", "A96", "A97", "A98", "A99"], "Opening": "Dutch"}
{"Moves": "1. e4", "ECO": ["B00"], "Opening": "King's pawn Opening"}
{"Moves": "1. e4 d5", "ECO": ["B01"], "Opening": "Scandinavian (centre counter) defence"}
{"Moves": "1. e4 Nf6", "ECO": ["B02", "B03", "B04", "B05"], "Opening": "Alekhine's defence"}
{"Moves": "1. e4 g6", "ECO": ["B06"], "Opening": "Robatsch (modern) defence"}
{"Moves": "1. e4 d6 2. d4 Nf6 3. Nc3", "ECO": ["B07", "B08", "B09"], "Opening": "Pirc defence"}
{"Moves": "1. e4 c6", "ECO": ["B10", "B11", "B12", "B13", "B14", "B15", "B16", "B17", "B18", "B19"], "Opening": "Caro-Kann defence"}
{"Moves": "1. e4 c5", "ECO": ["B20", "B21", "B22", "B23", "B24", "B25", "B26", "B27", "B28", "B29", "B30", "B31", "B32", "B33", "B34", "B35", "B36", "B37", "B38", "B39", "B40", "B41", "B42", "B43", "B44", "B45", "B46", "B47", "B48", "B49", "B50", "B51", "B52", "B53", "B54", "B55", "B56", "B57", "B58", "B59", "B60", "B61", "B62", "B63", "B64", "B65", "B66", "B67", "B68", "B69", "B70", "B71", "B72", "B73", "B74", "B75", "B76", "B77", "B78", "B79", "B80", "B81", "B82", "B83", "B84", "B85", "B86", "B87", "B88", "B89", "B90", "B91", "B92", "B93", "B94", "B95", "B96", "B97", "B98", "B99"], "Opening": "Sicilian defence"}
{"Moves": "1. e4 e6", "ECO": ["C00", "C01", "C02", "C03", "C04", "C05", "C06", "C07", "C08", "C09", "C10", "C11", "C12", "C13", "C14", "C15", "C16", "C17", "C18", "C19"], "Opening": "French defence"}
{"Moves": "1. e4 e5", "ECO": ["C20"], "Opening": "King's pawn game"}
{"Moves": "1. e4 e5 2. d4 exd4", "ECO": ["C21", "C22"], "Opening": "Centre game"}
{"Moves": "1. e4 e5 2. Bc4", "ECO": ["C23", "C24"], "Opening": "Bishop's Opening"}
{"Moves": "1. e4 e5 2. Nc3", "ECO": ["C25", "C26", "C27", "C28", "C29"], "Opening": "Vienna game"}
{"Moves": "1. e4 e5 2. f4", "ECO": ["C30", "C31", "C32", "C33", "C34", "C35", "C36", "C37", "C38", "C39"], "Opening": "King's gambit"}
{"Moves": "1. e4 e5 2. Nf3", "ECO": ["C40"], "Opening": "King's knight Opening"}
{"Moves": "1. e4 e5 2. Nf3 d6", "ECO": ["C41"], "Opening": "Philidor's defence"}
{"Moves": "1. e4 e5 2. Nf3 Nf6", "ECO": ["C42", "C43"], "Opening": "Petrov's defence"}
{"Moves": "1. e4 e5 2. Nf3 Nc6", "ECO": ["C44"], "Opening": "King's pawn game"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. d4 exd4 4. Nxd4", "ECO": ["C45"], "Opening": "Scotch game"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. Nc3", "ECO": ["C46"], "Opening": "Three knights game"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. Nc3 Nf6 4. d4", "ECO": ["C47", "C48", "C49"], "Opening": "Four knights, Scotch variation"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. Bc4", "ECO": ["C50"], "Opening": "Italian Game"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. Bc4 Bc5 4. b4", "ECO": ["C51", "C52"], "Opening": "Evans gambit"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. Bc4 Bc5 4. c3", "ECO": ["C53", "C54"], "Opening": "Giuoco Piano"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. Bc4 Nf6", "ECO": ["C55", "C56", "C57", "C58", "C59"], "Opening": "Two knights defence"}
{"Moves": "1. e4 e5 2. Nf3 Nc6 3. Bb5", "ECO": ["C60", "C61", "C62", "C63", "C64", "C65", "C66", "C67", "C68", "C69", "C70", "C71", "C72", "C73", "C74", "C75", "C76", "C77", "C78", "C79", "C80", "C81", "C82", "C83", "C84", "C85", "C86", "C87", "C88", "C89", "C90", "C91", "C92", "C93", "C94", "C95", "C96", "C97", "C98", "C99"], "Opening": "Ruy Lopez (Spanish Opening)"}
{"Moves": "1. d4 d5", "ECO": ["D00"], "Opening": "Queen's pawn game"}
{"Moves": "1. d4 d5 2. Nc3 Nf6 3. Bg5", "ECO": ["D01"], "Opening": "Richter-Veresov attack"}
{"Moves": "1. d4 d5 2. Nf3", "ECO": ["D02"], "Opening": "Queen's pawn game"}
{"Moves": "1. d4 d5 2. Nf3 Nf6 3. Bg5", "ECO": ["D03"], "Opening": "Torre attack (Tartakower variation)"}
{"Moves": "1. d4 d5 2. Nf3 Nf6 3. e3", "ECO": ["D04", "D05"], "Opening": "Queen's pawn game"}
{"Moves": "1. d4 d5 2. c4", "ECO": ["D06"], "Opening": "Queen's Gambit"}
{"Moves": "1. d4 d5 2. c4 Nc6", "ECO": ["D07", "D08", "D09"], "Opening": "Queen's Gambit Declined, Chigorin defence"}
{"Moves": "1. d4 d5 2. c4 c6", "ECO": ["D10", "D11", "D12", "D13", "D14", "D15"], "Opening": "Queen's Gambit Declined Slav defence"}
{"Moves": "1. d4 d5 2. c4 c6 3. Nf3 Nf6 4. Nc3 dxc4 5. a4", "ECO": ["D16"], "Opening": "Queen's Gambit Declined Slav accepted, Alapin variation"}
{"Moves": "1. d4 d5 2. c4 c6 3. Nf3 Nf6 4. Nc3 dxc4 5. a4 Bf5", "ECO": ["D17", "D18", "D19"], "Opening": "Queen's Gambit Declined Slav, Czech defence"}
{"Moves": "1. d4 d5 2. c4 dxc4", "ECO": ["D20", "D21", "D22", "D23", "D24", "D25", "D26", "D27", "D28", "D29"], "Opening": "Queen's gambit accepted"}
{"Moves": "1. d4 d5 2. c4 e6", "ECO": ["D30", "D31", "D32", "D33", "D34", "D35", "D36", "D37", "D38", "D39", "D40", "D41", "D42"], "Opening": "Queen's gambit declined"}
{"Moves": "1. d4 d5 2. c4 e6 3. Nc3 Nf6 4. Nf3 c6", "ECO": ["D43", "D44", "D45", "D46", "D47", "D48", "D49"], "Opening": "Queen's Gambit Declined semi-Slav"}
{"Moves": "1. d4 d5 2. c4 e6 3. Nc3 Nf6 4. Bg5", "ECO": ["D50", "D51", "D52", "D53", "D54", "D55", "D56", "D57", "D58", "D59", "D60", "D61", "D62", "D63", "D64", "D65", "D66", "D67", "D68", "D69"], "Opening": "Queen's Gambit Declined, 4.Bg5"}
{"Moves": "1. d4 Nf6 2. c4 g6 3. f3 d5", "ECO": ["D70", "D71", "D72", "D73", "D74", "D75", "D76", "D77", "D78", "D79"], "Opening": "Neo-Gruenfeld defence"}
{"Moves": "1. d4 Nf6 2. c4 g6 3. Nc3 d5", "ECO": ["D80", "D81", "D82", "D83", "D84", "D85", "D86", "D87", "D88", "D89", "D90", "D91", "D92", "D93", "D94", "D95", "D96", "D97", "D98", "D99"], "Opening": "Gruenfeld defence"}
{"Moves": "1. d4 Nf6 2. c4 e6", "ECO": ["E00"], "Opening": "Queen's pawn game"}
{"Moves": "1. d4 Nf6 2. c4 e6 3. g3 d5 4. Bg2", "ECO": ["E01", "E02", "E03", "E04", "E05", "E06", "E07", "E08", "E09"], "Opening": "Catalan, closed"}
{"Moves": "1. d4 Nf6 2. c4 e6 3. Nf3", "ECO": ["E10"], "Opening": "Queen's pawn game"}
{"Moves": "1. d4 Nf6 2. c4 e6 3. Nf3 Bb4+", "ECO": ["E11"], "Opening": "Bogo-Indian defence"}
{"Moves": "1. d4 Nf6 2. c4 e6 3. Nf3 b6", "ECO": ["E12", "E13", "E14", "E15", "E16", "E17", "E18", "E19"], "Opening": "Queen's Indian defence"}
{"Moves": "1. d4 Nf6 2. c4 e6 3. Nc3 Bb4", "ECO": ["E20", "E21", "E22", "E23", "E24", "E25", "E26", "E27", "E28", "E29", "E30", "E31", "E32", "E33", "E34", "E35", "E36", "E37", "E38", "E39", "E40", "E41", "E42", "E43", "E44", "E45", "E46", "E47", "E48", "E49", "E50", "E51", "E52", "E53", "E54", "E55", "E56", "E57", "E58", "E59"], "Opening": "Nimzo-Indian defence"}
{"Moves": "1. d4 Nf6 2. c4 g6", "ECO": ["E60", "E61", "E62", "E63", "E64", "E65", "E66", "E67", "E68", "E69", "E70", "E71", "E72", "E73", "E74", "E75", "E76", "E77", "E78", "E79", "E80", "E81", "E82", "E83", "E84", "E85", "E86", "E87", "E88", "E89", "E90", "E91", "E92", "E93", "E94", "E95", "E96", "E97", "E98", "E99"], "Opening": "King's Indian defence"}"""

In [ ]:
from io import StringIO

ECO_codes = pd.read_json(StringIO(eco_data), lines=True)

ECO_codes

In [ ]:
codes_exploded = ECO_codes.explode('ECO')

def find_eco(move_seq):
    for index, row in codes_exploded.iterrows():
        if move_seq.startswith(row['Moves']):
                    return row['ECO']
    return None

In [ ]:
# Identify the indices where ECO is null
missing_eco_mask = df['ECO'].isnull()

# Assign the results directly to the original dataframe
df.loc[missing_eco_mask, 'ECO'] = df.loc[missing_eco_mask, 'Moves'].apply(find_eco)

In [ ]:
df.ECO.isnull().sum()

## Data Transformation

### ECO
- Simplifying the eco code to 3 as they are official documentation about it.
- Can be used to create a opening column

In [ ]:
df.ECO.value_counts()

In [ ]:
df.ECO.str.slice(0, 3).value_counts()

In [ ]:
df.ECO = df.ECO.str.slice(0, 3)
df.ECO

### Result
- Changing 1/2-1/2: Draw
- Changing 1-0: White
- Changing 0-1: Black

In [ ]:
df.Result.value_counts()

In [ ]:
df['Result'] = df['Result'].map({'1/2-1/2': 'Draw', '1-0': 'White', '0-1': 'Black'})

In [ ]:
df.Result.value_counts()

## Type Columns 

In [ ]:
df.dtypes

### WhiteElo and BlackElo
- changing type from int to uint16 as the max value is near 3000 and cant be negative 

In [ ]:
df[['WhiteElo','BlackElo']].describe()

In [ ]:
df[['WhiteElo','BlackElo']] = df[['WhiteElo','BlackElo']].astype('uint16')

### BlackTitle and WhiteTitle
- changing type from object to categorical as the unique values are limited

In [ ]:
df.WhiteTitle.unique()

In [ ]:
df.WhiteTitle.unique()

In [ ]:
df[['WhiteTitle','BlackTitle']] = df[['WhiteTitle','BlackTitle']].astype('category')

In [ ]:
df.head()

# Feature Engineering

In [ ]:
df.head()

### Opening from ECO
- We convert the openning code to opening names

(The ECO Codes is a classification system for the chess openings moves in which openings are divided in five volumes labeled from "A" through "E".)

In [ ]:
df.ECO.value_counts()

In [ ]:
eco_map = dict(zip(codes_exploded['ECO'], codes_exploded['Opening']))
find_eco = lambda eco_code: eco_map.get(eco_code, np.nan)

df['Opening'] = df.ECO.apply(find_eco)

df.Opening.isnull().sum()

## Date to Year
- Converting the Date column to year, as it simplifies analysis and month and day subset of date have way to many null values.

In [ ]:
df.Date

In [ ]:
df['Date'].str.contains('\\?').sum()

In [ ]:
for i, period in enumerate(['Year','Month','Day']):
    print(period, df['Date'].str.split('.').str.get(i).str.contains('\\?').sum())

In [ ]:
df['Year'] = pd.to_numeric(df['Date'].str.split('.').str.get(0), errors='coerce')
df = df.dropna(subset=['Year'])
df.loc[:, 'Year'] = df['Year'].astype('uint16')

df.Year.describe()

# Reorder and select Columns for analysis

In [ ]:
df.columns

In [ ]:
new_order = [
    # Metadata
    'Event', 'Site', 'Year', 'Round', 
    
    # Players
    'White', 'WhiteTitle', 'WhiteElo', 
    'Black', 'BlackTitle', 'BlackElo', 
    
    # Game Data
    'Result', 'ECO', 'Opening', 'PlyCount', 'Moves'
]

df = df[new_order]

df.head()

# Univariant analysis